In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/test_predicted_fingerprints.npy
/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet


In [3]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 55.1 MB/s  0:00:00 eta 0:00:01


# 1. Importaciones, Configuración de Entorno y Rutas

In [4]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from tqdm import tqdm

# --- CONFIGURACIÓN DE RUTAS ---
DATA_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TRAIN_FOLDS_PATH = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet'

# Dataset donde guardaste los pesos del modelo del Notebook 2
MODEL_DIR = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints'
MODEL_PATH = os.path.join(MODEL_DIR, 'fingerprint_mlp_fold0.pth')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_FOLD = 0  # Usaremos el Fold 0 como conjunto de validación

# 2. Definición de Arquitectura y Dataset de Validación

In [5]:
class SpectrumMLP(nn.Module):
    def __init__(self, input_dim=2001, output_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

class MS2ValDataset(Dataset):
    def __init__(self, df, bin_size=0.5, max_mz=1000.0):
        self.df = df.reset_index(drop=True)
        self.bin_size = bin_size
        self.max_mz = max_mz
        self.num_bins = int(max_mz / bin_size)  # 2000 bins

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mzs = row.get('ms2_mzs', [])
        ints = row.get('ms2_normalized_intensities', [])
        precursor_mz = row.get('precursor_mz', 0.0)
        
        spectrum_vector = np.zeros(self.num_bins, dtype=np.float32)
        
        if mzs is not None and ints is not None and len(mzs) > 0 and len(ints) > 0:
            mzs_arr = np.nan_to_num(np.array(mzs, dtype=np.float32), nan=0.0)
            ints_arr = np.nan_to_num(np.array(ints, dtype=np.float32), nan=0.0)
            
            valid_mask = (mzs_arr >= 0) & (mzs_arr < self.max_mz)
            mzs_arr = mzs_arr[valid_mask]
            ints_arr = ints_arr[valid_mask]
            
            if len(mzs_arr) > 0:
                bins = (mzs_arr / self.bin_size).astype(int)
                bins = np.clip(bins, 0, self.num_bins - 1)
                np.maximum.at(spectrum_vector, bins, ints_arr)
        
        # Concatenar precursor_mz / 1000.0 (total = 2001 features)
        x = np.append(spectrum_vector, np.float32(precursor_mz / 1000.0))
        return torch.tensor(x, dtype=torch.float32)

# 3. Cargar Folds y Separar Validación

In [8]:
print(f"Cargando {TRAIN_FOLDS_PATH}...")
df_all = pd.read_parquet(TRAIN_FOLDS_PATH)

col_smiles = 'normalized_smiles' if 'normalized_smiles' in df_all.columns else 'smiles'

# Asegurar que tenemos un identificador único por molécula (usamos inchikey14 o inchikey)
id_col = 'inchikey14' if 'inchikey14' in df_all.columns else 'inchikey'

# Separar el fold de validación local (Fold 0)
val_df = df_all[df_all['fold'] == EVAL_FOLD].copy().reset_index(drop=True)
print(f"Espectros de validación en Fold {EVAL_FOLD}: {len(val_df):,}")

# Identificar moléculas únicas en validación usando el identificador químico
unique_val_mols = val_df[id_col].unique()
val_ground_truth = val_df.groupby(id_col)[col_smiles].first().to_dict()
val_precursor_mzs = val_df.groupby(id_col)['precursor_mz'].first().to_dict()

print(f"Moléculas únicas en validación: {len(unique_val_mols):,}")

Cargando /kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet...
Espectros de validación en Fold 0: 507,922
Moléculas únicas en validación: 55,162


# 4. Inferencia sobre el Fold de Validación con el Modelo Entrenado

In [10]:
print("Cargando modelo neuronal entrenado en Fase 2...")
val_dataset = MS2ValDataset(val_df)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2)

model = SpectrumMLP(input_dim=2001, output_dim=1024).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

pred_fps_list = []
with torch.no_grad():
    for x_batch in tqdm(val_loader, desc="Inferencia de Fingerprints"):
        x_batch = x_batch.to(DEVICE)
        preds = model(x_batch)
        pred_fps_list.append(preds.cpu().numpy())

val_preds = np.concatenate(pred_fps_list, axis=0)

# Agrupar y promediar fingerprints por molecule_id
val_df['fp_idx'] = np.arange(len(val_df))
mol_to_fp_val = {}

for mol_id, group in val_df.groupby(id_col):
    indices = group['fp_idx'].values
    avg_fp = val_preds[indices].mean(axis=0)
    mol_to_fp_val[mol_id] = (avg_fp > 0.5).astype(np.uint8)

del val_preds
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("✓ Predicciones de validación listas.")

Cargando modelo neuronal entrenado en Fase 2...


Inferencia de Fingerprints: 100%|██████████| 1985/1985 [00:37<00:00, 52.81it/s]


✓ Predicciones de validación listas.


# 5: Construcción de la Matriz de Candidatos Completa

In [11]:
print("Construyendo librería de candidatos desde el dataset completo...")
candidate_df = df_all.drop_duplicates(subset=['inchikey14']).reset_index(drop=True)

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
num_candidates = len(candidate_df)
candidate_fps_matrix = np.zeros((num_candidates, 1024), dtype=np.uint8)
candidate_mzs = np.zeros(num_candidates, dtype=np.float32)
valid_smiles = []

valid_count = 0
for idx, row in tqdm(candidate_df.iterrows(), total=num_candidates, desc="Indexando Candidatos"):
    smi = str(row[col_smiles]) if pd.notnull(row[col_smiles]) else ""
    if not smi:
        continue
    
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = morgan_gen.GetFingerprint(mol)
        candidate_fps_matrix[valid_count] = np.frombuffer(fp.ToBitString().encode('utf-8'), 'u1') - ord('0')
        candidate_mzs[valid_count] = row.get('precursor_mz', 0.0)
        valid_smiles.append(smi)
        valid_count += 1

candidate_fps_matrix = candidate_fps_matrix[:valid_count]
candidate_mzs = candidate_mzs[:valid_count]
candidate_df = pd.DataFrame({'smiles': valid_smiles, 'precursor_mz': candidate_mzs})

print(f"✓ Total candidatos indexados: {len(candidate_df):,}")

Construyendo librería de candidatos desde el dataset completo...


Indexando Candidatos: 100%|██████████| 275810/275810 [02:02<00:00, 2258.33it/s]

✓ Total candidatos indexados: 275,810


# 6: Cálculo de la Métrica Oficial MRR@25 con Filtrado por Masa

In [12]:
def batch_tanimoto_similarity(query_fp, candidate_matrix):
    cand_sums = candidate_matrix.sum(axis=1)
    query_sum = query_fp.sum()
    intersection = np.dot(candidate_matrix, query_fp)
    union = cand_sums + query_sum - intersection
    scores = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)
    return np.nan_to_num(scores, nan=0.0)

def compute_mrr_at_25(query_mol_ids, mol_to_fp, mol_to_mz, ground_truth, candidate_fps, candidate_mzs, candidate_df, mz_tolerance=0.5):
    reciprocal_ranks = []
    top_k = 25
    top1_hits = 0
    top5_hits = 0
    top25_hits = 0

    for mol_id in tqdm(query_mol_ids, desc="Evaluando MRR@25"):
        true_smiles = ground_truth[mol_id]
        pred_bit_array = mol_to_fp[mol_id]
        target_mz = mol_to_mz[mol_id]
        
        # Filtro de masa ±0.5 Da
        mz_mask = np.abs(candidate_mzs - target_mz) <= mz_tolerance
        filtered_indices = np.where(mz_mask)[0]
        
        # Si la ventana es muy estrecha, ampliar a ±2.0 Da
        if len(filtered_indices) < top_k:
            mz_mask = np.abs(candidate_mzs - target_mz) <= 2.0
            filtered_indices = np.where(mz_mask)[0]
            
        if len(filtered_indices) == 0:
            reciprocal_ranks.append(0.0)
            continue
            
        sub_matrix = candidate_fps[filtered_indices]
        scores = batch_tanimoto_similarity(pred_bit_array, sub_matrix)
        
        # Extraer Top-25
        fetch_k = min(top_k, len(scores))
        top_sub_indices = np.argpartition(scores, -fetch_k)[-fetch_k:]
        top_sub_indices = top_sub_indices[np.argsort(scores[top_sub_indices])[::-1]]
        
        retrieved_smiles = candidate_df.iloc[filtered_indices[top_sub_indices]]['smiles'].tolist()
        
        # Verificar si el SMILES verdadero está en las predicciones
        if true_smiles in retrieved_smiles:
            rank = retrieved_smiles.index(true_smiles) + 1  # Rango 1-indexed
            reciprocal_ranks.append(1.0 / rank)
            if rank == 1:
                top1_hits += 1
            if rank <= 5:
                top5_hits += 1
            if rank <= 25:
                top25_hits += 1
        else:
            reciprocal_ranks.append(0.0)
            
    mrr_score = np.mean(reciprocal_ranks)
    total = len(query_mol_ids)
    
    print("\n" + "="*45)
    print("      RESULTADOS DE VALIDACIÓN LOCAL (MRR@25)     ")
    print("="*45)
    print(f"MRR@25 Score Final : {mrr_score:.5f}")
    print(f"Top-1  Accuracy   : {top1_hits / total * 100:.2f}% ({top1_hits}/{total})")
    print(f"Top-5  Accuracy   : {top5_hits / total * 100:.2f}% ({top5_hits}/{total})")
    print(f"Top-25 Accuracy   : {top25_hits / total * 100:.2f}% ({top25_hits}/{total})")
    print("="*45)
    
    return mrr_score

# Ejecutar evaluación sobre las moléculas de validación
mrr = compute_mrr_at_25(
    query_mol_ids=unique_val_mols,
    mol_to_fp=mol_to_fp_val,
    mol_to_mz=val_precursor_mzs,
    ground_truth=val_ground_truth,
    candidate_fps=candidate_fps_matrix,
    candidate_mzs=candidate_mzs,
    candidate_df=candidate_df,
    mz_tolerance=0.5
)

Evaluando MRR@25: 100%|██████████| 55162/55162 [02:02<00:00, 451.51it/s]


      RESULTADOS DE VALIDACIÓN LOCAL (MRR@25)     
MRR@25 Score Final : 0.01324
Top-1  Accuracy   : 0.40% (223/55162)
Top-5  Accuracy   : 1.81% (997/55162)
Top-25 Accuracy   : 7.76% (4281/55162)
